In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import pandas as pd
import json
from adaptive_latents import datasets

In [ ]:
import traceback

In [ ]:
import psutil
import os

pid = os.getpid()
process = psutil.Process(pid)

with open("asdf.txt", "w+") as fhan:
    for frame in traceback.extract_stack():
        fhan.write(str(frame) + "\n")
    fhan.write(process.name() + "\n")

In [ ]:
experiment_t = datasets.Odoherty21Dataset().neural_data.t

In [ ]:
# with open('profile_prosvd.txt', 'r') as fhan:
with open('profile_prosvd_old_just_e.txt', 'r') as fhan:
    dicts = []
    for line in fhan:
        dicts.append(json.loads(line))

In [ ]:
df = pd.DataFrame.from_dict(dicts)
df['input_samples'] = df['repeat_type']
df.loc[df.input_samples != 'minimal', 'input_samples'] = df.loc[df.input_samples != 'minimal', 'input_samples'] * len(experiment_t)
df.loc[df.input_samples == 'minimal', 'input_samples'] = 11

df['input_time'] = df['input_samples'] * np.diff(experiment_t).mean()

df['repeats'] = df['repeat_type']
df.loc[df.repeats=='minimal', 'repeats'] = 11/len(experiment_t)

df

In [ ]:
plt.hist(df[df.repeat_type==2].t, bins=100);

In [ ]:
%matplotlib inline
fig, ax = plt.subplots()

x_axis = 'repeats'

for i, f in enumerate(sorted(df['sampled_function'].unique())):
    sub_df = df[df['sampled_function'] == f]
    sub_df.plot.scatter(x_axis, 't', ax=ax, color=f'C{i}')
    x = np.array(sub_df[x_axis].unique())[:,None]
    y = np.array([sub_df[sub_df[x_axis]==rt].t.min() for rt in x[:,0]])[:,None]
    lr = LinearRegression()
    lr.fit(x, y)
    xlim = np.array(ax.get_xlim())
    ax.set_autoscale_on(False)
    ax.plot(xlim, lr.predict(xlim[:,None]), label=f'{f}: m={lr.coef_[0,0]:.2f}, b={lr.intercept_[0]:.2f}', ls='--')

ax.axvline(0, color='k')
ax.legend()
